<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day09-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 9, Segment 3 discussion — L1 vs. L2, sparsity, by hand

The book page's regularization section applies **L2** (weight decay) to
the real classifier via PyTorch's `weight_decay` optimizer argument, and
mentions in passing that **L1** tends to push some weights to *exactly*
zero rather than just shrinking all of them. That claim isn't checked
in the main notebook. Here, check it directly with a small linear
regression, comparing L1-penalized and L2-penalized fits on the exact
same synthetic data.

**Before running anything, discuss with your group:** if a feature is
genuinely useless (its true coefficient is 0, and it's just noise),
which penalty do you expect to actually zero out that feature's learned
weight: L1, L2, both, or neither? Write a guess, then run the cells
below.

In [1]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

# 10 input features: only the first 3 actually matter (true weights below);
# the other 7 are pure noise with true weight 0.
n_samples, n_features = 200, 10
true_w = np.array([2.0, -1.5, 1.0, 0, 0, 0, 0, 0, 0, 0])
X = np.random.randn(n_samples, n_features)
y = X @ true_w + 0.1 * np.random.randn(n_samples)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

def fit(penalty, l1_lambda=0.0, weight_decay=0.0, epochs=2000, lr=0.05):
    torch.manual_seed(0)
    model = nn.Linear(n_features, 1, bias=False)
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    for _ in range(epochs):
        opt.zero_grad()
        pred = model(X_t)
        loss = nn.functional.mse_loss(pred, y_t)
        if l1_lambda > 0:
            loss = loss + l1_lambda * model.weight.abs().sum()
        loss.backward()
        opt.step()
    return model.weight.detach().numpy().flatten()

w_plain = fit("none")
w_l1 = fit("l1", l1_lambda=0.05)
w_l2 = fit("l2", weight_decay=0.05)

print(f"{'feature':8}{'true':>8}{'no reg':>10}{'L1':>10}{'L2':>10}")
for i in range(n_features):
    print(f"x{i:<7}{true_w[i]:>8.2f}{w_plain[i]:>10.4f}{w_l1[i]:>10.4f}{w_l2[i]:>10.4f}")

n_exact_zero_l1 = np.sum(np.abs(w_l1[3:]) < 1e-3)
n_exact_zero_l2 = np.sum(np.abs(w_l2[3:]) < 1e-3)
print()
print(f"Of the 7 truly-useless features, L1 drove {n_exact_zero_l1} to (near-)exactly zero.")
print(f"Of the 7 truly-useless features, L2 drove {n_exact_zero_l2} to (near-)exactly zero.")

feature     true    no reg        L1        L2
x0          2.00    1.9960    1.9660    1.9404
x1         -1.50   -1.5045   -1.4814   -1.4691
x2          1.00    0.9927    0.9625    0.9615
x3          0.00   -0.0047    0.0001   -0.0004
x4          0.00    0.0018   -0.0023   -0.0061
x5          0.00    0.0059   -0.0008    0.0081
x6          0.00    0.0056    0.0004    0.0087
x7          0.00   -0.0063   -0.0004   -0.0008
x8          0.00    0.0046   -0.0019   -0.0032
x9          0.00    0.0045    0.0022    0.0022

Of the 7 truly-useless features, L1 drove 4 to (near-)exactly zero.
Of the 7 truly-useless features, L2 drove 2 to (near-)exactly zero.


**Discuss the real result against your prediction.** L1's penalty
(`lambda * sum(|w|)`) has a constant-magnitude pull toward zero
regardless of how small a weight already is, so it can push a
genuinely useless weight all the way to (near) zero. L2's penalty
(`lambda * sum(w^2)`) pulls harder on large weights and progressively
weaker as a weight approaches zero, so it shrinks everything but rarely
zeroes anything out exactly — you should see the noise features' L2
weights are small but still visibly nonzero, unlike L1's.

**Connect back:** the book's `day06.qmd` page (Appendix A material) made
this same claim in the abstract ("L1 tries to estimate the median... L2
the mean") — this is the same idea made concrete with real numbers, the
same kind of independent verification this course does with every
worked example. One group presents which features L1 zeroed out and
whether that matches the features that were actually noise (`x3`
through `x9`).